# Cara Kerja Algoritma XGBoost — Penelusuran 1 Sampel dari Awal Sampai Akhir

Notebook ini menelusuri **satu komentar nyata** melewati seluruh tahapan pipeline XGBoost yang sudah
kamu latih (TF-IDF Bigram → Chi-Square → XGBoost), dan mengambil **angka asli** dari model
terlatih (bukan angka ilustratif) di setiap tahap -- supaya penjelasan cara kerja algoritma di
skripsimu didukung bukti komputasi nyata, bukan hanya contoh hipotetis.

**Struktur notebook:**
1. Muat model & vectorizer terlatih (hasil `modelling_xgb_lr_indobert_balanced_copy.ipynb`)
2. Tentukan komentar contoh + preprocessing
3. Ekstraksi fitur TF-IDF (Bigram) -- fitur aktif & bobotnya (ASLI dari vectorizer)
4. Seleksi fitur Chi-Square -- fitur yang lolos seleksi (ASLI)
5. Inisialisasi f₀(x) -- base score model
6. Hitung Gradient & Hessian iterasi pertama (dihitung manual dari rumus, dengan p dari model)
7. Bongkar pohon ke-1 hasil training ASLI (`get_dump`) -- split, threshold, Gain sungguhan
8. Lacak akumulasi skor mentah (raw margin) per checkpoint iterasi (1, 10, 100, semua pohon)
9. Softmax & Prediksi Kelas akhir
10. Ringkasan tabel semua tahap (siap salin ke skripsi, kolom Tahap/Proses/Hasil)

## 1. Setup & Muat Model Terlatih

In [1]:
import pandas as pd
import numpy as np
import pickle, os, re
import xgboost as xgb

# ── Sesuaikan path ini dengan lokasi project kamu ────────────────────────────
BASE_DIR   = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'
MODEL_DIR  = os.path.join(BASE_DIR, 'models', 'ml_v3')
PREP_DIR   = os.path.join(BASE_DIR, 'data', 'processed')

# Pilih skenario & model yang mau ditelusuri -- default: XGBoost Tuned A-Gabungan
NAMA_MODEL = 'XGB_v2_Tuned_A_Gabungan'

LABEL_MAP   = {'keluhan': 0, 'saran': 1, 'pujian': 2}
INV_MAP     = {v: k for k, v in LABEL_MAP.items()}
CLASS_NAMES = ['Keluhan', 'Saran', 'Pujian']

model_path = os.path.join(MODEL_DIR, f'{NAMA_MODEL}.pkl')
vec_path   = os.path.join(MODEL_DIR, f'vec_{NAMA_MODEL}.pkl')

model = pickle.load(open(model_path, 'rb'))
vec   = pickle.load(open(vec_path, 'rb'))
tfidf, selector = vec['tfidf'], vec['selector']

booster = model.get_booster()
n_estimators = model.n_estimators
print(f'✅ Model dimuat: {NAMA_MODEL}')
print(f'   n_estimators : {n_estimators}')
print(f'   max_depth    : {model.max_depth}')
print(f'   learning_rate: {model.learning_rate}')
print(f'   Dimensi fitur setelah Chi-Square: {selector.k}')

✅ Model dimuat: XGB_v2_Tuned_A_Gabungan
   n_estimators : 500
   max_depth    : 8
   learning_rate: 0.2
   Dimensi fitur setelah Chi-Square: 8000


## 2. Komentar Contoh & Preprocessing

Ganti `KOMENTAR_CONTOH` di bawah sesuai komentar yang ingin kamu telusuri (bisa komentar nyata dari
dataset, atau contoh buatan seperti yang sudah kamu tulis di tabel). Notebook akan mencoba mencari
teks yang sudah dipreprocessing (`text_v2`) dari dataset kamu terlebih dahulu -- kalau ketemu, versi
ASLI hasil pipeline preprocessing-mu yang dipakai (paling akurat). Kalau tidak ketemu (misal komentar
buatan sendiri untuk ilustrasi), notebook memakai fungsi preprocessing sederhana sebagai pengganti --
**ganti fungsi `preprocessing_sederhana()` dengan fungsi preprocessing ASLI kamu (yang mencakup
NEGASI_PROTECT, STEM_PROTECT, dan stemming Sastrawi) supaya hasilnya konsisten dengan skripsi.**

In [ ]:
KOMENTAR_CONTOH = "Aplikasi BPJS sering error saat mau daftar antrian, tolong diperbaiki."
LABEL_AKTUAL = 'keluhan'

# ── Coba cari di dataset preprocessed yang sudah ada ─────────────────────────
teks_preprocessed = None
try:
    df_check = pd.read_csv(os.path.join(PREP_DIR, 'data_preprocessed_v2.csv'))
    match = df_check[df_check['text'].astype(str).str.contains(
        re.escape(KOMENTAR_CONTOH[:30]), case=False, na=False, regex=True)]
    if len(match) > 0:
        teks_preprocessed = match.iloc[0]['text_v2']
        print(f'✅ Ditemukan di dataset asli! text_v2 = "{teks_preprocessed}"')
except FileNotFoundError:
    pass

if teks_preprocessed is None:
    print('⚠️  Komentar tidak ditemukan di dataset -- pakai preprocessing sederhana sebagai gantinya.')
    print('   GANTI fungsi di bawah dengan fungsi preprocessing ASLI kamu untuk hasil yang konsisten!')

    def preprocessing_sederhana(text):
        text = text.lower()
        text = re.sub(r'[^a-z\s]', ' ', text)          # buang simbol/angka
        stopwords_min = {'saat', 'mau', 'yang', 'di', 'ke', 'dan', 'nya'}
        tokens = [t for t in text.split() if t not in stopwords_min and len(t) > 2]
        return ' '.join(tokens)

    teks_preprocessed = preprocessing_sederhana(KOMENTAR_CONTOH)
    print(f'   Hasil preprocessing sederhana: "{teks_preprocessed}"')

print(f'\nKomentar asli   : "{KOMENTAR_CONTOH}"')
print(f'Label aktual (y): {LABEL_AKTUAL} (kelas {LABEL_MAP[LABEL_AKTUAL]})')
print(f'Setelah preproc : "{teks_preprocessed}"')

## 3. Ekstraksi Fitur TF-IDF (Bigram) -- Nilai ASLI dari Vectorizer Terlatih

In [ ]:
X_tfidf = tfidf.transform([teks_preprocessed])
feature_names_all = np.array(tfidf.get_feature_names_out())

nonzero_idx = X_tfidf.nonzero()[1]
nonzero_vals = X_tfidf.toarray()[0][nonzero_idx]
urutan = np.argsort(-nonzero_vals)

print(f'Dimensi vektor TF-IDF penuh : {X_tfidf.shape[1]:,}')
print(f'Jumlah fitur bernilai tidak nol untuk komentar ini: {len(nonzero_idx)}')
print(f'\nFitur aktif (diurutkan dari bobot tertinggi):')
print(f'{"Fitur":30s} {"Bobot TF-IDF":>12s}')
print('-' * 44)
for i in urutan:
    idx = nonzero_idx[i]
    print(f'{feature_names_all[idx]:30s} {nonzero_vals[i]:>12.4f}')

## 4. Seleksi Fitur Chi-Square -- Fitur yang Lolos Seleksi (ASLI)

In [ ]:
X_selected = selector.transform(X_tfidf)
mask_selected = selector.get_support()
feature_names_selected = feature_names_all[mask_selected]

# Fitur aktif dari komentar ini yang LOLOS seleksi Chi-Square
nonzero_idx_sel = X_selected.nonzero()[1]
nonzero_vals_sel = X_selected.toarray()[0][nonzero_idx_sel]

print(f'Dimensi setelah Chi-Square (k_best): {X_selected.shape[1]:,} (dari {X_tfidf.shape[1]:,} fitur TF-IDF)')
print(f'\nDari fitur aktif komentar ini, yang LOLOS seleksi Chi-Square:')
if len(nonzero_idx_sel) == 0:
    print('  (Tidak ada fitur dari komentar ini yang lolos seleksi -- komentar terlalu pendek/generik)')
else:
    urutan_sel = np.argsort(-nonzero_vals_sel)
    print(f'{"Fitur":30s} {"Bobot TF-IDF":>12s}   {"Skor Chi2"}')
    print('-' * 60)
    chi2_scores = selector.scores_[mask_selected]
    for i in urutan_sel:
        idx = nonzero_idx_sel[i]
        fname = feature_names_selected[idx]
        skor_chi2 = chi2_scores[idx]
        print(f'{fname:30s} {nonzero_vals_sel[i]:>12.4f}   {skor_chi2:>10.2f}')

## 5. Inisialisasi Prediksi Awal f₀(x) -- Base Score

In [ ]:
dmatrix = xgb.DMatrix(X_selected)

# Raw margin score SEBELUM pohon manapun ditambahkan (base_score, iterasi ke-0)
margin_awal = booster.predict(dmatrix, output_margin=True, iteration_range=(0, 0))
print(f'Skor mentah f₀(x) sebelum ada pohon: {margin_awal[0]}')

# Kalau XGBoost tidak bisa predict dengan 0 iterasi, base_score biasanya seragam (mendekati 0 / log-odds basis)
def softmax(z):
    e = np.exp(z - np.max(z))
    return e / e.sum()

p_awal = softmax(margin_awal[0]) if margin_awal[0].size == 3 else np.array([1/3, 1/3, 1/3])
print(f'Setelah softmax -> P awal = {np.round(p_awal, 4)} ({CLASS_NAMES})')

## 6. Hitung Gradient & Hessian Iterasi Pertama

Dihitung manual memakai rumus turunan cross-entropy softmax standar XGBoost:
gₖ = pₖ − yₖ (one-hot), hₖ = pₖ(1−pₖ)

In [ ]:
y_true_idx = LABEL_MAP[LABEL_AKTUAL]
y_onehot = np.zeros(3); y_onehot[y_true_idx] = 1

g = p_awal - y_onehot
h = p_awal * (1 - p_awal)

print(f'p (probabilitas awal)      : {np.round(p_awal, 4)}')
print(f'y (one-hot, label aktual)  : {y_onehot} ({LABEL_AKTUAL})')
print(f'g (gradient) = p - y       : {np.round(g, 4)}')
print(f'h (hessian)  = p(1-p)      : {np.round(h, 4)}')
print(f'\n(Catatan: ini gradien/hessian KONTRIBUSI dari 1 sampel ini saja. Nilai G dan H yang')
print(f' dipakai XGBoost untuk menentukan split pada Tahap 7 adalah PENJUMLAHAN gradien/hessian')
print(f' dari SELURUH sampel data latih yang berada di node yang sama, bukan hanya sampel ini.)')

## 7. Bongkar Pohon ke-1 Hasil Training ASLI

Menggunakan `booster.get_dump()` untuk melihat struktur pohon pertama yang SUNGGUHAN dipelajari
model dari seluruh data latih -- termasuk fitur yang dipilih untuk split dan nilai Gain aslinya.

In [ ]:
tree_dump = booster.get_dump(with_stats=True)[0]
print('Struktur pohon ke-1 (kelas Keluhan, node pertama beberapa level):')
print(tree_dump[:2000])
print('...' if len(tree_dump) > 2000 else '')

# Cari tahu leaf mana yang ditempati sampel ini pada pohon ke-1
leaf_index = booster.predict(dmatrix, pred_leaf=True, iteration_range=(0, 1))
print(f'\nSampel ini jatuh ke leaf index (pohon ke-1, per kelas): {leaf_index[0]}')

print('\nCatatan interpretasi: nilai "gain" pada dump di atas adalah Gain SUNGGUHAN yang dipakai')
print('XGBoost saat memutuskan split terbaik pada setiap node, dihitung dari rumus:')
print('Gain = 1/2 [G_L^2/(H_L+lambda) + G_R^2/(H_R+lambda) - (G_L+G_R)^2/(H_L+H_R+lambda)] - gamma')
print('dengan G_L, H_L, G_R, H_R adalah JUMLAH gradien/hessian seluruh sampel train di cabang tsb.')

## 8. Lacak Akumulasi Skor Mentah (Raw Margin) per Checkpoint Iterasi

Menunjukkan bagaimana skor mentah untuk ketiga kelas berubah secara bertahap seiring bertambahnya
jumlah pohon (dari model SUNGGUHAN, bukan simulasi).

In [ ]:
import matplotlib.pyplot as plt

checkpoints = sorted(set([1, 5, 10, 25, 50, 100, 150, 200, n_estimators] +
                          [n_estimators // 4, n_estimators // 2]))
checkpoints = [c for c in checkpoints if 0 < c <= n_estimators]

riwayat_margin = []
for cp in checkpoints:
    m = booster.predict(dmatrix, output_margin=True, iteration_range=(0, cp))
    riwayat_margin.append(m[0])

riwayat_margin = np.array(riwayat_margin)
print(f'{"Iterasi ke-":>12s}   {"Keluhan":>10s} {"Saran":>10s} {"Pujian":>10s}')
for cp, m in zip(checkpoints, riwayat_margin):
    print(f'{cp:>12d}   {m[0]:>10.4f} {m[1]:>10.4f} {m[2]:>10.4f}')

plt.figure(figsize=(8, 5))
for i, kelas in enumerate(CLASS_NAMES):
    plt.plot(checkpoints, riwayat_margin[:, i], marker='o', label=kelas)
plt.xlabel('Jumlah Pohon (Iterasi)')
plt.ylabel('Skor Mentah (Raw Margin)')
plt.title(f'Akumulasi Skor Mentah per Iterasi\nKomentar: "{KOMENTAR_CONTOH[:50]}..."')
plt.axhline(0, color='gray', linewidth=0.7)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('akumulasi_skor_xgboost.png', dpi=150)
plt.show()
print('\n✅ Grafik disimpan -> akumulasi_skor_xgboost.png')

## 9. Softmax & Prediksi Kelas Akhir

In [ ]:
margin_final = booster.predict(dmatrix, output_margin=True)[0]
p_final = softmax(margin_final)
pred_idx = int(np.argmax(p_final))
pred_label = INV_MAP[pred_idx]

print(f'Skor mentah akhir (setelah {n_estimators} pohon):')
print(f'  Keluhan = {margin_final[0]:.4f}   Saran = {margin_final[1]:.4f}   Pujian = {margin_final[2]:.4f}')
print(f'\nSetelah Softmax:')
print(f'  P(Keluhan) = {p_final[0]:.4f}   P(Saran) = {p_final[1]:.4f}   P(Pujian) = {p_final[2]:.4f}')
print(f'\nPrediksi kelas (argmax)  : {pred_label}')
print(f'Label aktual             : {LABEL_AKTUAL}')
print(f'Status                   : {"✅ BENAR" if pred_label == LABEL_AKTUAL else "❌ SALAH"}')

## 10. Ringkasan Tabel Semua Tahap (Siap Salin ke Skripsi)

In [ ]:
ringkasan = pd.DataFrame([
    ['1', 'Input Data (x, y)', f'x = "{KOMENTAR_CONTOH}"; y = {LABEL_AKTUAL}'],
    ['2', 'Preprocessing', f'"{teks_preprocessed}"'],
    ['3', 'Ekstraksi Fitur TF-IDF', f'{len(nonzero_idx)} fitur aktif dari {X_tfidf.shape[1]:,} dimensi'],
    ['4', 'Seleksi Fitur Chi-Square', f'{len(nonzero_idx_sel)} fitur lolos dari {X_selected.shape[1]:,} fitur terpilih'],
    ['5', 'Inisialisasi f0(x)', f'P awal = {np.round(p_awal, 3).tolist()}'],
    ['6', 'Gradient & Hessian (iterasi 1)', f'g = {np.round(g, 3).tolist()}, h = {np.round(h, 3).tolist()}'],
    ['7', 'Pohon ke-1 (struktur asli)', 'Lihat dump pohon pada Tahap 7'],
    ['8', 'Akumulasi skor per iterasi', f'Skor akhir = {np.round(margin_final, 3).tolist()}'],
    ['9', 'Softmax', f'P = {np.round(p_final, 3).tolist()}'],
    ['10', 'Prediksi Kelas', f'{pred_label} ({"benar" if pred_label==LABEL_AKTUAL else "salah"} vs label aktual {LABEL_AKTUAL})'],
], columns=['Tahap', 'Proses', 'Hasil'])

pd.set_option('display.max_colwidth', 100)
display(ringkasan)
ringkasan.to_csv('ringkasan_tahapan_xgboost.csv', index=False)
print('\n✅ Tabel ringkasan tersimpan -> ringkasan_tahapan_xgboost.csv (siap dibuka di Excel/Word)')